# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** Use `@id` identifiers for referencing all dataset entities, such as record sets and fields.

In [ ]:
# Explore record sets and fields
print('Record Sets and Fields:')
for record_set in metadata.record_sets:
    print(f"Record Set Name: {record_set.name}\n  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
    print()

# For demonstration, preview a few records from the first record set
if metadata.record_sets:
    first_rs_id = metadata.record_sets[0].id
    print(f'Example records from RecordSet @id: {first_rs_id}')
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if i > 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> In this example, we extract all record sets found in the metadata.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
    else:
        dataframes[record_set] = pd.DataFrame()

# Use the first record set for demonstration
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Loaded DataFrame columns for RecordSet @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> You must reference all field names by their `@id`. Check the overview above for the available field `@id`s.

**Tip:** We use the first numeric field we find for demonstration, and a grouping field if available.

In [ ]:
# Identify record set and a numeric field for analysis
main_df = dataframes[main_record_set_id]

# Find a numeric field to analyze
numeric_field_id = None
group_field_id = None
for field in metadata.record_sets[0].fields:
    if field.data_type in ('Integer', 'Float', 'Number') and numeric_field_id is None:
        # Try to pick a field that likely matches real data
        if field.id in main_df.columns:
            numeric_field_id = field.id
    # Optionally select a grouping field (string/categorical)
    if group_field_id is None and field.data_type == 'Text' and field.id in main_df.columns:
        group_field_id = field.id

if numeric_field_id:
    # Some columns may be strings, convert to float if necessary
    col_dtype = main_df[numeric_field_id].dtype
    # Try to convert if needed
    if not pd.api.types.is_numeric_dtype(col_dtype):
        main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
    print(f"Applying threshold at {threshold:.2f} on field {numeric_field_id}.")
    # Filter based on threshold
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Below is an example using matplotlib to plot the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id:
        # Boxplot by group
        plt.figure(figsize=(10,4))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load metadata and records from the FAIR² Croissant dataset using the `mlcroissant` library.
- We explored the available record sets and field structure (referenced by their `@id`).
- We extracted tabular data from a record set, performed filtering and normalization on a numeric field, and visualized its distribution.
- These steps can be adapted to more specialized analyses as required for downstream research or machine learning workflows.